# Interactive UMAP Explorer — Amine Fingerprints

2-D UMAP projection of the amines used in the BSH conjugation screen.
Bubble size = number of enzymes with positive activity for that amine.
Hover shows the molecular structure via **molplotly**.

In [ ]:
# ── Configuration ────────────────────────────────────────────
from pathlib import Path

# Fingerprint settings (swap FP_TYPE to compare projections)
FP_TYPE    = "morgan"          # "morgan" | "rdkit" | "topological_torsion" | "atom_pair"
FP_RADIUS  = 2                 # only used by morgan
FP_NBITS   = 1024

# Activity threshold – an enzyme counts as "positive" when Intensity > this value
INTENSITY_THRESHOLD = 0.0

# UMAP hyper-parameters
UMAP_N_NEIGHBORS  = 8
UMAP_MIN_DIST     = 0.3
UMAP_METRIC       = "jaccard"   # natural for binary fingerprints
UMAP_RANDOM_STATE = 42

# Paths (relative to notebooks/)
DATA_DIR   = Path("../data")
OUTPUT_DIR = Path("../outputs")
REACTANTS_FILE = DATA_DIR / "bsh_reactants_SMILES_corrected.xlsx"
ENUM_FILE  = OUTPUT_DIR / "swap_enumeration_FINAL.xlsx"
HEAT_FILE  = OUTPUT_DIR / "ipsita_heatmap_long.csv"

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize

import umap

import plotly.express as px
from IPython.display import display

try:
    import molplotly
    HAS_MOLPLOTLY = True
except ImportError:
    HAS_MOLPLOTLY = False
    print("molplotly not installed – hover will show text only. "
          "Install with: pip install molplotly")

RDLogger.DisableLog('rdApp.warning')

In [ ]:
# ── Fingerprint helpers ──────────────────────────────────────

def standardize_smiles(smi: str) -> str | None:
    """Strip salts, normalize, canonicalize."""
    if pd.isna(smi) or not str(smi).strip():
        return None
    mol = Chem.MolFromSmiles(str(smi).strip())
    if mol is None:
        return None
    try:
        mol = rdMolStandardize.Normalize(mol)
        mol = rdMolStandardize.LargestFragmentChooser().choose(mol)
        mol = SaltRemover().StripMol(mol, dontRemoveEverything=True)
        Chem.SanitizeMol(mol)
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return Chem.MolToSmiles(mol, canonical=True)


def compute_fingerprint(mol, fp_type: str = "morgan",
                        radius: int = 2, n_bits: int = 1024) -> np.ndarray:
    """Return a binary numpy fingerprint vector for *mol*."""
    if mol is None:
        return np.zeros(n_bits, dtype=np.uint8)

    fp_type = fp_type.lower()
    if fp_type == "morgan":
        bv = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    elif fp_type == "rdkit":
        bv = Chem.RDKFingerprint(mol, fpSize=n_bits)
    elif fp_type == "topological_torsion":
        from rdkit.Chem import rdMolDescriptors
        bv = rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(mol, nBits=n_bits)
    elif fp_type == "atom_pair":
        from rdkit.Chem import rdMolDescriptors
        bv = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=n_bits)
    else:
        raise ValueError(f"Unknown fp_type: {fp_type!r}")

    arr = np.zeros(n_bits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(bv, arr)
    return arr

In [ ]:
# ── Load data & build amine summary ──────────────────────────

# 1. All amines in the reaction mix (master list)
reactants_df = pd.read_excel(REACTANTS_FILE)
type_col = "Compount Type" if "Compount Type" in reactants_df.columns else "Compound Type"
all_amines = (
    reactants_df[reactants_df[type_col].str.lower().str.contains("amine", na=False)]
    [["Compound_Name", "SMILES"]]
    .rename(columns={"Compound_Name": "Amine_Name", "SMILES": "Amine_SMILES"})
    .drop_duplicates(subset=["Amine_Name"])
    .reset_index(drop=True)
)
print(f"Amines in reaction mix: {len(all_amines)}")

# 2. Enumeration file: ProductName -> (Amine_Name, Amine_SMILES)
enum_df = pd.read_excel(ENUM_FILE)
prod_to_amine = (
    enum_df[["ProductName", "Amine_Name", "Amine_SMILES"]]
    .drop_duplicates(subset=["ProductName"])
)

# Amine type from enumeration
amine_type_map = (
    enum_df[["Amine_Name", "Amine_Type"]]
    .dropna(subset=["Amine_Name"])
    .drop_duplicates(subset=["Amine_Name"])
    .set_index("Amine_Name")["Amine_Type"]
    .to_dict()
)

# 3. Heatmap: (Enzyme, ProductName, Intensity)
heat = pd.read_csv(HEAT_FILE)
heat["Intensity"] = pd.to_numeric(heat["Intensity"], errors="coerce").fillna(0.0)

# Join heatmap to enumeration to get amine identity
merged = heat.merge(prod_to_amine, on="ProductName", how="inner")

# 4. Count positive enzymes per amine (from heatmap)
pos = merged[merged["Intensity"] > INTENSITY_THRESHOLD]
activity = (
    pos.groupby("Amine_Name")["Enzyme"]
    .nunique()
    .reset_index()
    .rename(columns={"Enzyme": "n_positive_enzymes"})
)

# Total enzymes tested (same for all amines — full panel)
n_enzymes_tested = heat["Enzyme"].nunique()

# 5. Merge: all reaction mix amines + their activity counts
amine_df = all_amines.merge(activity, on="Amine_Name", how="left")
amine_df["n_positive_enzymes"] = amine_df["n_positive_enzymes"].fillna(0).astype(int)
amine_df["has_activity"] = amine_df["n_positive_enzymes"] > 0
amine_df["Activity"] = amine_df["has_activity"].map({True: "Active", False: "No activity"})

# Amine type: from enumeration if available, else classify from SMILES
_pat_primary   = Chem.MolFromSmarts("[NX3;H2;!$(NC=O)]")
_pat_secondary = Chem.MolFromSmarts("[NX3;H1;!$(NC=O)]")

def _classify_amine(row):
    # Check enumeration first
    known = amine_type_map.get(row["Amine_Name"])
    if pd.notna(known) and str(known).strip():
        return str(known).strip()
    # Fall back to SMILES-based classification
    smi = row.get("Amine_SMILES")
    if pd.isna(smi) or not str(smi).strip():
        return "unknown"
    mol = Chem.MolFromSmiles(str(smi).strip())
    if mol is None:
        return "unknown"
    if mol.HasSubstructMatch(_pat_primary):
        return "primary"
    if mol.HasSubstructMatch(_pat_secondary):
        return "secondary"
    return "other"

amine_df["Amine_Type"] = amine_df.apply(_classify_amine, axis=1)

# Summary label for hover
amine_df["hover_summary"] = amine_df.apply(
    lambda r: f"{r['Amine_Type']} amine | {r['n_positive_enzymes']}/{n_enzymes_tested} enzymes active",
    axis=1,
)

amine_df = amine_df.sort_values("n_positive_enzymes", ascending=False).reset_index(drop=True)

n_active = amine_df["has_activity"].sum()
n_inactive = len(amine_df) - n_active
print(f"With activity:    {n_active}")
print(f"No activity:      {n_inactive}")
print(f"Total plotted:    {len(amine_df)}")
print(f"Enzymes tested:   {n_enzymes_tested}")
amine_df

In [ ]:
# ── Compute fingerprints + UMAP ─────────────────────────────

# Standardize SMILES (strip salts like .Cl)
amine_df["SMILES_clean"] = amine_df["Amine_SMILES"].apply(standardize_smiles)
amine_df["Mol"] = amine_df["SMILES_clean"].apply(
    lambda s: Chem.MolFromSmiles(s) if s else None
)

valid = amine_df[amine_df["Mol"].notna()].copy().reset_index(drop=True)
print(f"Valid molecules: {len(valid)} / {len(amine_df)}")

# Fingerprint matrix
fp_matrix = np.stack([
    compute_fingerprint(mol, fp_type=FP_TYPE, radius=FP_RADIUS, n_bits=FP_NBITS)
    for mol in valid["Mol"]
])
print(f"FP matrix: {fp_matrix.shape}  (type={FP_TYPE}, bits={FP_NBITS})")

# UMAP
reducer = umap.UMAP(
    n_neighbors=min(UMAP_N_NEIGHBORS, len(valid) - 1),
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=UMAP_RANDOM_STATE,
)
embedding = reducer.fit_transform(fp_matrix)

valid["UMAP_1"] = embedding[:, 0]
valid["UMAP_2"] = embedding[:, 1]
print(f"UMAP done — {valid['has_activity'].sum()} active, "
      f"{(~valid['has_activity']).sum()} inactive.")

In [ ]:
# ── Molecule images for fallback hover (base64 PNGs) ────────
import io, base64

def mol_to_base64_png(mol, size=(200, 200)):
    if mol is None:
        return ""
    img = Draw.MolToImage(mol, size=size)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

valid["img_b64"] = valid["Mol"].apply(mol_to_base64_png)
print(f"Generated {valid['img_b64'].str.len().gt(0).sum()} molecule PNGs.")

In [ ]:
# ── Plot 1: Enzyme activity UMAP (bubble size + color = # positive enzymes) ──

# Only active amines for this plot
active_valid = valid[valid["has_activity"]].copy().reset_index(drop=True)

fig1 = px.scatter(
    active_valid,
    x="UMAP_1",
    y="UMAP_2",
    size="n_positive_enzymes",
    color="n_positive_enzymes",
    color_continuous_scale="Viridis",
    hover_name="Amine_Name",
    hover_data=["Amine_Type", "n_positive_enzymes", "SMILES_clean"],
    title=f"UMAP — Enzyme Activity per Amine ({FP_TYPE}, {FP_NBITS} bits)",
    labels={
        "n_positive_enzymes": "# positive enzymes",
        "Amine_Type": "Amine type",
        "SMILES_clean": "SMILES",
        "UMAP_1": "UMAP 1",
        "UMAP_2": "UMAP 2",
    },
)
fig1.update_layout(width=900, height=700, dragmode="pan")

if HAS_MOLPLOTLY:
    try:
        app1 = molplotly.add_molecules(
            fig=fig1,
            df=active_valid.reset_index(drop=True),
            smiles_col="SMILES_clean",
            title_col="Amine_Name",
            caption_cols=["Amine_Type", "n_positive_enzymes"],
            show_coords=False,
        )
        try:
            app1.run(jupyter_mode="inline", port=8701, jupyter_height=750)
        except (AttributeError, TypeError):
            app1.run_server(mode="inline", port=8701, height=750)
    except (Exception, SystemExit) as e:
        print(f"molplotly could not render — falling back to plain Plotly.")
        fig1.show()
else:
    fig1.show()

In [ ]:
# ── Plot 2: Active vs No Activity UMAP (all amines from reaction mix) ────────

valid["bubble_size"] = valid["n_positive_enzymes"].clip(lower=5)

fig2 = px.scatter(
    valid,
    x="UMAP_1",
    y="UMAP_2",
    size="bubble_size",
    color="Activity",
    color_discrete_map={"Active": "#2196F3", "No activity": "#E0E0E0"},
    hover_name="Amine_Name",
    hover_data=["Amine_Type", "n_positive_enzymes", "SMILES_clean"],
    title=f"UMAP — Active vs No Activity, All Reaction Mix Amines ({FP_TYPE}, {FP_NBITS} bits)",
    labels={
        "n_positive_enzymes": "# positive enzymes",
        "Amine_Type": "Amine type",
        "SMILES_clean": "SMILES",
        "UMAP_1": "UMAP 1",
        "UMAP_2": "UMAP 2",
    },
)
fig2.update_layout(width=900, height=700, dragmode="pan")

if HAS_MOLPLOTLY:
    try:
        app2 = molplotly.add_molecules(
            fig=fig2,
            df=valid.reset_index(drop=True),
            smiles_col="SMILES_clean",
            title_col="Amine_Name",
            color_col="Activity",
            caption_cols=["Amine_Type", "n_positive_enzymes"],
            show_coords=False,
        )
        try:
            app2.run(jupyter_mode="inline", port=8702, jupyter_height=750)
        except (AttributeError, TypeError):
            app2.run_server(mode="inline", port=8702, height=750)
    except (Exception, SystemExit) as e:
        print(f"molplotly could not render — falling back to plain Plotly.")
        fig2.show()
else:
    fig2.show()

In [ ]:
# ── Static molecule grid (for export / quick reference) ─────
import math

# Active amines
active_df = valid[valid["has_activity"]].sort_values("n_positive_enzymes", ascending=False)
inactive_df = valid[~valid["has_activity"]].sort_values("Amine_Name")

def _draw_grid(df, title):
    if df.empty:
        print(f"{title}: none")
        return
    mols = df["Mol"].tolist()
    legends = [
        f"{row.Amine_Name}\n({row.n_positive_enzymes} enzymes)"
        for _, row in df.iterrows()
    ]
    n_cols = min(6, len(mols))
    img = Draw.MolsToGridImage(
        mols, molsPerRow=n_cols, subImgSize=(280, 280),
        legends=legends, useSVG=False,
    )
    print(f"{title} ({len(mols)} amines)")
    display(img)

_draw_grid(active_df, "Active amines")
_draw_grid(inactive_df, "Amines with no detected activity")

## Usage Notes

**Changing the fingerprint type:** Edit `FP_TYPE` in the config cell and re-run the notebook.
Supported values: `"morgan"`, `"rdkit"`, `"topological_torsion"`, `"atom_pair"`.

**Adjusting the UMAP projection:**
- `UMAP_N_NEIGHBORS` — lower values emphasize local structure; higher values give a more global view.
- `UMAP_MIN_DIST` — controls how tightly points are packed.
- `UMAP_METRIC` — `"jaccard"` is the natural choice for binary fingerprints.

**Activity threshold:** Set `INTENSITY_THRESHOLD` to count only enzymes above a
specific intensity value as positive. Default is 0 (any nonzero activity).